In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

# easy to change settings
IMG_SIZE = 64
BATCH = 64
EPOCHS = 8

# load and prep dataset function
def prep_data(ds, is_train=False):
    # resize and normalize the images
    ds = ds.map(lambda x: (
        tf.cast(tf.image.resize(x["image"], (IMG_SIZE, IMG_SIZE)), tf.float32) / 255.0, 
        tf.cast(x["attributes"]["Smiling"], tf.float32) # getting the smiling label
    ), num_parallel_calls=tf.data.AUTOTUNE)
    
    if is_train:
        ds = ds.shuffle(5000)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

# build my cnn using sequential to save space!
my_model = tf.keras.Sequential([
    tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    tf.keras.layers.RandomFlip("horizontal"), # simple data augmentation
    
    tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(2),
    tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
    tf.keras.layers.MaxPooling2D(2),
    
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation="sigmoid") # sigmoid for smiling or not
])

my_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
my_model.summary()

print("\n--- loading dataset ---")
# loading celeb_a like the assignment says
(t_raw, v_raw, test_raw) = tfds.load("celeb_a", split=["train", "validation", "test"])

train_ds = prep_data(t_raw, True)
val_ds = prep_data(v_raw)
test_ds = prep_data(test_raw)

print("training starting now!!")
hist = my_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

print("\ntesting on unseen data:")
lss, acc = my_model.evaluate(test_ds)
print(f"Results => loss: {lss:.4f}, accuracy: {acc * 100:.2f}%")

# plot accuracy real quick
plt.plot(hist.history['accuracy'], label='Train Acc')
plt.plot(hist.history['val_accuracy'], label='Val Acc')
plt.legend()
plt.savefig("training_plot.png")
print("saved plot as training_plot.png!")
